# KNN Validation
Validates genre embeddings

In [ ]:
import json
import numpy as np
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import umap
import matplotlib
import re
from sklearn.cluster import KMeans
from collections import Counter

## Config

In [ ]:
EMBEDDINGS_PATH = "./data/embeddings/genre_embeddings.json"
SONGS_PATH = "./data/csv/songs.csv"
K = 10
TOP_N = 50

QUERY_GENRES = [
    "pop rap", "death metal", "jazz",
    "tropical house", "classical", "edm",
    "blues", "reggaeton", "folk", "house"
]

SPOT_GENRES = ["pop rap", "death metal", "jazz", "rock", "classical", "edm"]

N_CLUSTERS = 30
MIN_CLUSTER_SIZE = 8

## 1. Load Embeddings

In [ ]:
with open(EMBEDDINGS_PATH) as f:
    raw = json.load(f)

genres = list(raw.keys())
token2idx = {g: i for i, g in enumerate(genres)}
X = np.array(list(raw.values()), dtype=np.float32)

print(f"{len(genres)} genres | dim={X.shape[1]}")

## 2. KNN

In [ ]:
nbrs = NearestNeighbors(n_neighbors=K+1, metric='cosine').fit(X)
distances, indices = nbrs.kneighbors(X)

def knn(query, k=5):
    if query not in token2idx:
        return f"'{query}' not in vocab"
    idx = token2idx[query]
    return [(genres[indices[idx][i]], 1 - distances[idx][i]) for i in range(1, k+1)]

for q in SPOT_GENRES:
    print(f"\n→ {q}")
    for name, sim in knn(q):
        print(f"   {name:<35} {sim:.3f}")

## 3. Macro-Genre Labels

In [ ]:
MACRO = {
    "hip hop":      ["hip hop","rap","trap","drill","crunk","pop rap","dwn trap","southern hip hop","bounce"],
    "pop":          ["pop","dance pop","electropop","synth-pop","teen pop","indie pop","dream pop"],
    "rock":         ["rock","alternative rock","indie rock","classic rock","punk","shoegaze","grunge","emo"],
    "electronic":   ["electronic","edm","house","techno","trance","dubstep","ambient","idm","synthwave","lo-fi"],
    "r&b / soul":   ["r&b","soul","funk","neo soul","gospel","urban contemporary"],
    "country":      ["country","americana","bluegrass","folk","singer-songwriter","alt-country"],
    "metal":        ["metal","heavy metal","death metal","black metal","thrash","doom","metalcore"],
    "jazz / blues": ["jazz","blues","bebop","swing","acid jazz","blues rock"],
    "classical":    ["classical","orchestral","opera","baroque","neoclassical"],
    "latin":        ["latin","reggaeton","salsa","bachata","cumbia","latin pop"],
    "world / folk": ["world","folk","celtic","african","reggae","ska","dancehall"],
}
macro_list = sorted(MACRO.keys())

def get_macro(genre):
    g = genre.lower()
    for macro, kws in MACRO.items():
        for kw in kws:
            if kw in g: return macro
    return None

## 4. UMAP Plot

In [ ]:
reducer     = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
emb2d       = reducer.fit_transform(X)

km          = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_ids = km.fit_predict(emb2d)

def is_genre_name(name):
    if len(name) > 35 or len(name) < 3: return False
    if re.search(r'\d', name): return False
    if name[0] in '-_': return False
    if any(c in name for c in [':', '!', '?']): return False
    return True

cluster_labels = {}
for ci in range(N_CLUSTERS):
    mask = np.where(cluster_ids == ci)[0]
    if len(mask) < MIN_CLUSTER_SIZE: continue
    names = [genres[i] for i in mask if is_genre_name(genres[i])]
    if names:
        cluster_labels[ci] = Counter(names).most_common(1)[0][0]

cmap_clusters = matplotlib.colormaps['tab20'].resampled(N_CLUSTERS)

fig, ax = plt.subplots(figsize=(16, 11), facecolor='#0f0f0f')
ax.set_facecolor('#0f0f0f')
for ci in range(N_CLUSTERS):
    mask = np.where(cluster_ids == ci)[0]
    if len(mask) == 0: continue
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1], c=[cmap_clusters(ci)], s=18, alpha=0.8, linewidths=0)
    if len(mask) >= MIN_CLUSTER_SIZE and ci in cluster_labels:
        cx, cy = emb2d[mask, 0].mean(), emb2d[mask, 1].mean()
        ax.annotate(cluster_labels[ci], (cx, cy), color='white', fontsize=7,
                    ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='#000000aa', edgecolor='none'))
ax.set_title('Genre Embedding Space (UMAP)', color='white', fontsize=13, pad=15)
ax.set_xlabel('PC1', color='#888888'); ax.set_ylabel('PC2', color='#888888')
ax.tick_params(colors='#555555')
for spine in ax.spines.values(): spine.set_color('#333333')
plt.tight_layout()
plt.show()

## 5. Cosine Similarity Heatmap

In [ ]:
q_genres = [q for q in QUERY_GENRES if q in token2idx]
n = len(q_genres)
sim_matrix = np.zeros((n, n))
for i, ga in enumerate(q_genres):
    va = X[token2idx[ga]]
    for j, gb in enumerate(q_genres):
        vb = X[token2idx[gb]]
        sim_matrix[i,j] = max(0, (np.dot(va, vb) / (np.linalg.norm(va)*np.linalg.norm(vb)+1e-9)))

fig, ax = plt.subplots(figsize=(12, 10), facecolor='#0f0f0f')
ax.set_facecolor('#0f0f0f')
im = ax.imshow(sim_matrix, cmap='plasma', vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(q_genres, rotation=45, ha='right', color='white', fontsize=11)
ax.set_yticks(range(n)); ax.set_yticklabels(q_genres, color='white', fontsize=11)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center',
                color='white' if sim_matrix[i,j] < 0.7 else 'black', fontsize=9)
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.ax.yaxis.set_tick_params(color='white', labelcolor='white')
ax.set_title('Cosine Similarity Between Genres', color='white', fontsize=15, pad=15)
plt.tight_layout()
plt.show()

## 6. KNN Neighbour Bar Charts

In [ ]:
def is_clean(name):
    if len(name) > 40: return False
    if any(c.isdigit() for c in name): return False
    return True

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0f0f0f')
axes = axes.flatten()

for ax, q in zip(axes, SPOT_GENRES):
    ax.set_facecolor('#111111')
    if q not in token2idx:
        ax.set_visible(False); continue
    idx = token2idx[q]

    neighbours = []
    for i in range(1, len(indices[idx])):
        name = genres[indices[idx][i]]
        if is_clean(name):
            neighbours.append((name, 1 - distances[idx][i]))
        if len(neighbours) == 7: break

    names = [n for n, _ in neighbours]
    sims  = [s for _, s in neighbours]

    ax.barh(names[::-1], sims[::-1], color=plt.cm.plasma(np.linspace(0.3, 0.9, len(names))))
    ax.set_xlim(0, 1)
    ax.set_title(f'→ {q}', color='white', fontsize=12, pad=8)
    ax.tick_params(colors='white', labelsize=9)
    ax.set_xlabel('cosine similarity', color='#888888', fontsize=9)
    for spine in ax.spines.values(): spine.set_color('#333333')

plt.suptitle('KNN Nearest Neighbours by Genre', color='white', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Query Any Genre

In [ ]:
query = "grunge"

if query in token2idx:
    print(f"Nearest neighbours to '{query}':\n")
    for name, sim in knn(query, k=10):
        print(f"  {name:<35} {sim:.3f}")
else:
    print(f"'{query}' not found. Similar genres in vocab:")
    print([g for g in genres if query.split()[0] in g][:10])

In [ ]:
def get_macro(genre):
    g = genre.lower()
    for macro, kws in MACRO.items():
        for kw in kws:
            if kw in g: return macro
    return None

# Neighbour Purity
purity_scores = []
for i, genre in enumerate(genres):
    macro = get_macro(genre)
    if macro is None: continue
    neighbours = [genres[indices[i][j]] for j in range(1, K+1)]
    matches = sum(1 for n in neighbours if get_macro(n) == macro)
    purity_scores.append(matches / K)

print(f"Neighbour Purity:         {np.mean(purity_scores):.3f}  (how often neighbours share the same macro-genre)")

genre_macros = np.array([get_macro(g) for g in genres])
labeled_idx  = np.where(genre_macros != None)[0]
X_labeled    = X[labeled_idx]
labels_arr   = genre_macros[labeled_idx]

sim_matrix_full = X_labeled @ X_labeled.T

intra, inter = [], []
for i in range(len(labeled_idx)):
    for j in range(i+1, len(labeled_idx)):
        sim = float(sim_matrix_full[i, j])
        if labels_arr[i] == labels_arr[j]:
            intra.append(sim)
        else:
            inter.append(sim)

print(f"Intra-cluster similarity: {np.mean(intra):.3f}  (similarity within same macro-genre, higher = better)")
print(f"Inter-cluster similarity: {np.mean(inter):.3f}  (similarity across macro-genres, lower = better)")
print(f"Intra/Inter ratio:        {np.mean(intra)/np.mean(inter):.2f}x  (higher = more separable clusters)")

# Per-class purity breakdown
print(f"\nPurity by macro-genre:")
for macro in sorted(MACRO.keys()):
    scores = []
    for i, genre in enumerate(genres):
        if get_macro(genre) != macro: continue
        neighbours = [genres[indices[i][j]] for j in range(1, K+1)]
        matches = sum(1 for n in neighbours if get_macro(n) == macro)
        scores.append(matches / K)
    if scores:
        print(f"  {macro:<20} {np.mean(scores):.3f}  (n={len(scores)})")